# 模块说明

- **功能描述**：负责对爬取原始数据整理清洗
- **输出结果**：
    - `data_clean/city_budget_clean.csv`：主要城市财政数据表
    - `data_clean/real_estate_clean.csv`：主要城市房地产数据表
    - `data_clean/city_all_clean.csv`：合并数据表

# 一、数据爬取
## （一）数据来源
> 因数据爬取代码篇幅较大，本模块不做代码展示，仅做简要说明。
### 1.主要数据来源
- **国家统计局** - 主要城市年度数据
  - 网址: https://data.stats.gov.cn/easyquery.htm?cn=E0105
  - 路径: 首页 >> 地区数据 >> 主要城市年度数据
  - 获取方式: 通过API接口自动化获取
  - 代码实现：`codes/fetch_data.py`
  - **最终输出**：`merged_raw.csv`（GDP等数据）和`real_estate_raw.csv`（房地产相关数据）两个原始数据合并文件

### 2.补充数据来源
国家统计局数据存在部分缺失（共11个数据点），通过以下渠道补全：
- [**中国统计信息网**](http://www.tjcn.org/tjgb/) - 各地统计公报
- **城市政府官网** - 统计局公开数据
- [**统计年鉴下载网**](https://web.xiaze.org/tjgb/) - 历年统计公报

| 城市 | 年份 | 缺失指标 | 补全来源 |
|------|------|----------|----------|
| 哈尔滨 | 2022 | 一般公共预算收入、支出 | 哈尔滨市政府预算执行报告 |
| 昆明 | 2009 | 一般公共预算收入、支出、储蓄存款 | 昆明市统计公报 |
| 拉萨 | 2006 | GDP、储蓄存款余额 | 拉萨市统计公报 |
| 拉萨 | 2010 | GDP、储蓄存款余额 | 拉萨市统计公报 |
| 拉萨 | 2012 | 储蓄存款余额 | 拉萨市统计公报 |
| 拉萨 | 2013 | 储蓄存款余额 | 拉萨市统计公报 |

### 3.数据补全方法
1. **自动化爬虫**: 使用Python爬虫从统计公报网站自动提取数据
2. **人工核验**: 为增强鲁棒性，若**爬取失败，则应用人工查询数据**。
3. **数据整合**: 将补全数据合并回原始数据集
4. **完整性验证**: 确保所有核心指标(income, expend, gdp)无缺失
5. **代码实现**：`codes/raw_data_get.py`

人工查询数据及来源如下：
```text
哈尔滨 2022年:
  - 一般公共预算收入: 262.2亿元
  - 一般公共预算支出: 1065.5亿元
  来源: 哈尔滨市政府《关于2022年预算执行情况和2023年预算草案的报告》

昆明 2009年:
  - 一般预算收入: 201.61亿元 (2,016,125万元)
  - 一般预算支出: 270.45亿元 (2,704,495万元)
  - 储蓄存款余额: 1922.92亿元
  来源: 昆明市2009年国民经济和社会发展统计公报

拉萨 2006年:
  - GDP: 102.39亿元
  - 城乡居民储蓄存款: 78.99亿元
  来源: 拉萨市2006年国民经济和社会发展统计公报

拉萨 2010年:
  - GDP: 178.91亿元
  - 城乡居民储蓄存款余额: 151.03亿元
  来源: 拉萨市2010年国民经济和社会发展统计公报

拉萨 2012年:
  - 人民币个人储蓄存款余额: 223.83亿元
  来源: 拉萨市2012年统计数据

拉萨 2013年:
  - 人民币个人储蓄存款余额: 272.12亿元
  来源: 拉萨市2013年国民经济和社会发展统计公报
```

## （二）数据说明
### 1.`merged_raw.csv`指标说明
- `city`: 城市名称（36个主要城市）
- `year`: 年度（2006-2024）
- `income`: 地方一般公共预算收入(亿元)
- `expend`: 地方一般公共预算支出(亿元)
- `gdp`: 地区生产总值(亿元)
- `deposit`: 住户存款余额(亿元)

### 2.`real_estate_raw.csv`指标说明
- `A0302`:房地产开发投资额（亿元）
- `A0309`:商品房销售面积（万平方米）
- `A030B`:商品房平均销售价格（元/平方米）
- `A030A`:住宅商品房销售面积（万平方米）
- `A030C`:住宅商品房平均销售价格（元/平方米）

# 二、数据清洗
## （一）原始数据的读取与格式规范
### 1.目标
分别对`merged_raw.csv`和`real_estate_raw.csv`处理缺失值并做格式规范，输出为`city_budget_clean.csv`和`real_estate_clean.csv`
### 2.缺失处理及说明
#### 问题1：2025年数据缺失
- **情况概述**：所有城市2025年数据均缺失。
- **解决方案**：客观未公布，直接删除。
#### 问题2：拉萨 数据大量缺失
- **情况概述**：其房地产相关数据大量缺失。
- **解决方案**：本文仅能**将拉萨排除出**房地产的相关分析。


In [1]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

DATA_RAW = "data_raw"
DATA_CLEAN = "data_clean"
# 读取原始文件
df_city_budget = pd.read_csv(os.path.join(DATA_RAW, "merged_raw.csv"), dtype={"city": str, 'year': str})
df_real_estate = pd.read_csv(os.path.join(DATA_RAW, "real_estate_raw.csv"), dtype={"city": str, 'year': str})

# 处理缺失 - 删除2025年空数据行
df_city_budget.loc[:, 'year'] = pd.to_datetime(df_city_budget['year']).dt.year
df_city_budget = df_city_budget[df_city_budget['year'] <= 2024].copy()
df_real_estate.loc[:, 'year'] = pd.to_datetime(df_real_estate['year']).dt.year
df_real_estate = df_real_estate[df_real_estate['year'] <= 2024].copy()

# 处理缺失 - 房地产数据剔除拉萨
df_real_estate = df_real_estate[df_real_estate['city'] != '拉萨'].copy()

# 检查数据
print('======= 主要城市财政数据表情况 ========')
print(f'城市数量：{df_city_budget['city'].nunique()}')
print(f'年份范围：{df_city_budget['year'].min()}年~{df_city_budget['year'].max()}年')
print('缺失情况：')
print(df_city_budget.isnull().sum())
print('======= 主要城市房地产数据表情况 ========')
print(f'城市数量：{df_real_estate['city'].nunique()}')
print(f'年份范围：{df_real_estate['year'].min()}年~{df_real_estate['year'].max()}年')
print('缺失情况：')
print(df_real_estate.isnull().sum())

======= 主要城市财政数据表情况 ========
城市数量：36
年份范围：2006年~2024年
缺失情况：
city       0
year       0
income     0
expend     0
deposit    0
gdp        0
dtype: int64
======= 主要城市房地产数据表情况 ========
城市数量：35
年份范围：2006年~2024年
缺失情况：
city     0
year     0
A0302    0
A0309    0
A030B    0
A030A    0
A030C    0
dtype: int64


### 3.数据规范
#### 情况1：房地产数据表指标不够直观
- **情况概述**：其指标名称均为A030**这类格式的，不便于后续处理。
- **解决方案**：替换为易读的英文名称
#### 情况2：数据预运算
- **情况概述**：后续多项分析任务均需要计算 预算缺口 (gap)及其占GDP比重 财政收入支出及GDP增长率等。
- **解决方案**：预先计算如下指标并存在数据表下
    - `gap`: **预算缺口** = expend - income (亿元)
    - `gap_to_gdp`: **预算缺口占GDP比重** = gap / gdp
    - `income_growth`: 财政收入年度增长率(%)
    - `expend_growth`: 财政支出年度增长率(%)
    - `gdp_growth`: GDP年度增长率(%)
#### 情况3：原始数据中无城市分组信息
- **情况概述**：原始数据并无标注其隶属哪个经济区、是否为一线。
- **解决方案**：建立映射并手动添加如下指标：
    - `region_group`: 区域分组（珠三角/长三角/其他）
    - `is_tier1`: 是否为一线城市（北上广深）
#### 情况4：部分城市数据离群值多
- **情况概述**：诸如拉萨，经测试发现，其数据波动较大。
- **解决方案**：因作业任务有极端分析，故此处**不作处理**。

In [2]:
# 数据规范 - 房地产指标名称替换
cols_dict = {'A0302': 're_invest', 'A0309': 'sale_area', 'A030B': 're_avg_price', 'A030A': 'res_sale_area', 'A030C': 'res_avg_price'}
df_real_estate = df_real_estate.rename(columns = cols_dict).copy()

# 数据规范 - 预运算
df_city_budget['gap'] = df_city_budget['expend'] - df_city_budget['income']
df_city_budget['gap_to_gdp'] = df_city_budget['gap'] / df_city_budget['gdp']
for col in ['income', 'expend']:
    df_city_budget[f'{col}_growth'] = df_city_budget.groupby('city')[col].pct_change() * 100

# 数据规范 - 城市分组
tier1 = ['北京', '上海', '广州', '深圳']
pearl_river = ['广州', '深圳', '珠海', '佛山', '东莞', '中山', '惠州']
yangtze_river = ['上海', '南京', '杭州', '苏州', '无锡', '宁波', '合肥', '南通']
def assign_group(city):
    if city in pearl_river:
        return '珠三角'
    elif city in yangtze_river:
        return '长三角'
    else:
        return '其他'
df_city_budget['region_group'] = df_city_budget['city'].apply(assign_group)
df_city_budget['is_tier1'] = df_city_budget['city'].isin(tier1)

# 检查数据
print(f'珠三角城市(数据中可用):{[c for c in pearl_river if c in df_city_budget['city'].unique()]}')
print(f"长三角城市(数据中可用): {[c for c in yangtze_river if c in df_city_budget['city'].unique()]}")

珠三角城市(数据中可用):['广州', '深圳']
长三角城市(数据中可用): ['上海', '南京', '杭州', '宁波', '合肥']


# 三、保存清洗后数据

In [3]:
# 保存主要城市财政数据
df_city_budget.to_csv(os.path.join(DATA_CLEAN, "city_budget_clean.csv"), index=False, encoding="utf-8-sig")
print(f"财政数据已清洗完毕并输出，数据表形状：{df_city_budget.shape}")
df_city_budget.head()

财政数据已清洗完毕并输出，数据表形状：(684, 12)


,city,year,income,expend,deposit,gdp,gap,gap_to_gdp,income_growth,expend_growth,region_group,is_tier1
0,上海,2006,1576.0742,1795.5660,8730.000000,10825.4,219.4918,0.020276,NaN,NaN,长三角,True
1,上海,2007,2074.4792,2181.6780,8745.220000,13179.8,107.1988,0.008134,31.623194,21.503637,长三角,True
2,上海,2008,2358.7464,2593.9161,11464.150000,14877.1,235.1697,0.015807,13.703063,18.895460,长三角,True
3,上海,2009,2540.2975,2989.6500,13707.320000,16181.4,449.3525,0.027770,7.696932,15.256234,长三角,True
4,上海,2010,2873.5840,3302.8862,15650.239121,18319.6,429.3022,0.023434,13.119979,10.477354,长三角,True


In [4]:

# 保存主要城市房地产数据
df_real_estate.to_csv(os.path.join(DATA_CLEAN, "real_estate_clean.csv"), index=False, encoding="utf-8-sig")
print(f"房地产数据已清洗完毕并输出，数据表形状：{df_real_estate.shape}")
df_real_estate.head()

房地产数据已清洗完毕并输出，数据表形状：(665, 7)


,city,year,re_invest,sale_area,re_avg_price,res_sale_area,res_avg_price
0,上海,2006,1275.5938,3025.4003,7196.000146,2615.4915,7039.000127
1,上海,2007,1307.5256,3694.9578,8360.999955,3279.1745,8252.999955
2,上海,2008,1435.7349,2339.2900,8195.000000,2007.4800,8115.000000
3,上海,2009,1462.0724,3372.4500,12840.000000,2928.0400,12364.000000
4,上海,2010,1980.6800,2060.9600,14464.000000,1690.8200,14290.000000


In [5]:
# 为后续可能的分析任务，合并保存两个数据表
df_all = df_city_budget.merge(df_real_estate, on=['city', 'year'], how='left')
df_all.to_csv(os.path.join(DATA_CLEAN, "city_all_clean.csv"), index=False, encoding="utf-8-sig")
print(f"合并数据已保存，数据表形状：{df_all.shape}")
df_all.head()

合并数据已保存，数据表形状：(684, 17)


,city,year,income,expend,deposit,gdp,gap,gap_to_gdp,income_growth,expend_growth,region_group,is_tier1,re_invest,sale_area,re_avg_price,res_sale_area,res_avg_price
0,上海,2006,1576.0742,1795.5660,8730.000000,10825.4,219.4918,0.020276,NaN,NaN,长三角,True,1275.5938,3025.4003,7196.000146,2615.4915,7039.000127
1,上海,2007,2074.4792,2181.6780,8745.220000,13179.8,107.1988,0.008134,31.623194,21.503637,长三角,True,1307.5256,3694.9578,8360.999955,3279.1745,8252.999955
2,上海,2008,2358.7464,2593.9161,11464.150000,14877.1,235.1697,0.015807,13.703063,18.895460,长三角,True,1435.7349,2339.2900,8195.000000,2007.4800,8115.000000
3,上海,2009,2540.2975,2989.6500,13707.320000,16181.4,449.3525,0.027770,7.696932,15.256234,长三角,True,1462.0724,3372.4500,12840.000000,2928.0400,12364.000000
4,上海,2010,2873.5840,3302.8862,15650.239121,18319.6,429.3022,0.023434,13.119979,10.477354,长三角,True,1980.6800,2060.9600,14464.000000,1690.8200,14290.000000
